# W11B · 9B · 리뷰 임베딩을 FM 입력에 붙여 학습하기
**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026-11-11**

예상 → 수계산 → 실행 → 한 곳 수정 → 이유 설명. `None`은 자신의 답이며 예제 변수와 분리돼 있습니다. Run all 성공과 문제 풀이 완료는 다릅니다. Colab에서 자신의 사본을 저장하세요.


In [ ]:
from __future__ import annotations
import os,sys,subprocess,json,hashlib
os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
os.environ['TF_ENABLE_ONEDNN_OPTS']='0'
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w11b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps,neural,llm] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.31.0'],check=True)
    for package,wanted in [('tensorflow','2.21.0'),('keras','3.15.1')]:
        if package in sys.modules and getattr(sys.modules[package],'__version__',None)!=wanted:
            raise RuntimeError('설치가 갱신되었습니다. 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.')
from pathlib import Path
from typing import Callable
import numpy as np
import pandas as pd
from IPython.display import display
from luna_recsys.llm_runtime import prepare_runtime,CHAT_MODEL,EMBED_MODEL,MODEL_DIGESTS
from luna_recsys.llm_recommendation import (load_amazon_reviews,select_cohort,split_fm_observations,
    prepare_prompt_case,recommendation_schema,run_prompt)
from luna_recsys.llm_explainer import build_llm_explainer,export_displayed_table
CACHE=Path.home()/'.cache/luna-recsys/w11b'
print('Python',sys.version.split()[0],'수업 ref',COURSE_VERSION,'CPU / 실제 모델 / API 키 불필요')


### 준비 · 최초 다운로드를 시작하세요
공식 실행 파일과 두 모델은 첫 준비 때 약2.2GB를 내려받습니다. CPU·디스크·네트워크에 따라 수 분이 걸립니다. 같은 런타임에서는 검증한 캐시를 재사용합니다. Colab 기본 CPU를 사용하세요. 실패하면 오류를 확인하고 이 셀을 다시 실행하며, 가짜 응답이나 임의 벡터로 바꾸지 않습니다.

교재는 Ollama/Llama3.1을 사용합니다. 수업은 무료 CPU에서 실행하려고 Qwen2.5 0.5B와 Nomic v1.5(768차원)를 고정합니다. 모델과 표본이 다르므로 교재 수치 재현이라고 부르지 않습니다.


In [ ]:
client=prepare_runtime(CACHE/'runtime')
frame=select_cohort(load_amazon_reviews(CACHE/'data'))
print('공식 자료 / 교재 제공 자료와 다른 소표본:',json.dumps(frame.attrs,ensure_ascii=False))
print('실제 모델 digest:',MODEL_DIGESTS)
display(frame[['user_id','item_id','rating','timestamp']].head())


In [ ]:
from luna_recsys.embedding_fm import (make_feature_recipe,fit_embedding_fm,predict_embedding_fm,
    evaluate_embedding_fm,toy_embedding_fm)
from luna_recsys.fm_core import _parameters,_input,pair_contributions
parts=split_fm_observations(frame)
print('분할:',parts['protocol'])
print('무작위 관측 분할입니다. 미래 시점·신규 사용자 일반화 시험은 아닙니다.')


### 본문 실행 정의 · `aggregate_reviews`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: 60f62c994f21defc87f95979d2cc8bd43f5e470c0cfdf4547d4a3013f49cce7b
def aggregate_reviews(frame: pd.DataFrame, by: str = 'item_id', max_reviews: int = 3,
                      max_chars: int = 300, positive_only: bool = False) -> dict:
    """Group only the supplied rows; retain source IDs and explicit text truncation.

    Returns texts/source_rows/omitted_chars by ID. No encoding, writes or learning.
    """
    if by not in ('user_id', 'item_id') or min(max_reviews, max_chars) < 1:
        raise ValueError('Choose a valid grouping key and positive text limits.')
    groups = frame.sort_values(['timestamp', 'row_id'], kind='stable').groupby(by, sort=True)
    texts, source_rows, omitted = {}, {}, {}
    for key, group in groups:
        chosen = group[group.rating.ge(4)] if positive_only else group
        chosen = chosen[chosen.text.fillna('').str.strip().ne('')].tail(max_reviews)
        original = chosen.text.astype(str).tolist()
        clipped = [text[:max_chars] for text in original]
        texts[key] = '\n'.join(clipped)
        source_rows[key] = chosen.row_id.astype(int).tolist()
        omitted[key] = sum(len(a) - len(b) for a, b in zip(original, clipped))
    return {'texts': texts, 'source_rows': source_rows, 'omitted_chars': omitted}


### 본문 실행 정의 · `normalize_embeddings`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: 3540bcc505bccee3361871fd548f5bc6bddad0cb2f9d589b6123a7b94f01a976
def normalize_embeddings(vectors, dimension: int = 8) -> np.ndarray:
    """Select a prefix then L2-normalize rows. Zero prefixes stay zero, not unit vectors.

    Input (N,D), output (N,dimension), new float array. This is not PCA or learned
    feature selection. Coordinates do not have fixed human-readable genre names.
    """
    values = np.asarray(vectors, dtype=float)
    if values.ndim != 2 or not 1 <= dimension <= values.shape[1] or not np.isfinite(values).all():
        raise ValueError('Need finite (N,D) values and 1<=dimension<=D.')
    prefix = values[:, :dimension].copy()
    norms = np.linalg.norm(prefix, axis=1, keepdims=True)
    return np.divide(prefix, norms, out=np.zeros_like(prefix), where=norms > 0)


### 본문 실행 정의 · `assemble_features`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: b45fbb0ab11952cad552fe723eafae778da8506627187542f6e609a5f7911a89
def assemble_features(user: str, item: str, recipe: dict, use_embeddings: bool = True) -> np.ndarray:
    """One observation -> [user one-hot,item one-hot,user vector,item vector].

    n=U+I+2d with embeddings, U+I without. Unknown IDs have zero ID blocks and
    training-mean vectors; their absence is separately reported by prediction.
    """
    n_users, n_items, d = len(recipe['users']), len(recipe['items']), recipe['dimension']
    out = np.zeros(n_users + n_items + (2*d if use_embeddings else 0))
    if user in recipe['user_index']:
        out[recipe['user_index'][user]] = 1.
    if item in recipe['item_index']:
        out[n_users + recipe['item_index'][item]] = 1.
    if use_embeddings:
        offset = n_users + n_items
        out[offset:offset+d] = recipe['user_vectors'].get(user, recipe['default_user'])
        out[offset+d:offset+2*d] = recipe['item_vectors'].get(item, recipe['default_item'])
    return out


### 본문 실행 정의 · `fm_score`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: cad016bc8e0b9941bfb72d485317f0d6c6f84fa95e3c0edaaf585306512342d7
def fm_score(x, w, V, bias: float = 0.) -> dict:
    """효율식으로 2-way FM을 분해한다. x,w:(n,), V:(n,k), bias:고정 실수.

    반환 dict: prediction/bias/linear/interaction은 실수, linear_terms:(n,),
    weighted_factors:(n,k), factor_sums/factor_self/factor_interactions:(k,).
    factor_self는 sum((v_if*x_i)**2)이며 sum(v_if**2*x_i)가 아니다.
    예: toy_parameters와 encode_observation의 기본값 -> prediction 3.25.
    """
    w, V, bias = _parameters(w, V, bias)
    x = _input(x, len(w))
    weighted = V * x[:, None]
    sums = weighted.sum(axis=0)
    self_terms = np.square(weighted).sum(axis=0)
    factors = .5 * (np.square(sums) - self_terms)
    linear_terms = w * x
    linear, interaction = float(linear_terms.sum()), float(factors.sum())
    prediction = bias + linear + interaction
    if not np.isfinite(prediction):
        raise ValueError('계산 범위를 넘었습니다. 입력/모수의 크기를 줄이세요.')
    return {'prediction': prediction, 'bias': bias, 'linear': linear,
            'interaction': interaction, 'linear_terms': linear_terms,
            'weighted_factors': weighted, 'factor_sums': sums,
            'factor_self': self_terms, 'factor_interactions': factors}


### 본문 실행 정의 · `embedding_fm_step`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: ef8c615f46eef18ab61abe6900bb09d00cd5f9bbb58842b4e8e742e1e2c3a3da
def embedding_fm_step(x, y: float, w, V, bias: float, learning_rate: float = .03,
                       predict_fn: Callable = fm_score) -> dict:
    """One simultaneous SGD update for L=.5*(y-yhat)^2, fixed bias and zero L2.

    x,w:(n,), V:(n,K). Gradient uses the same pre-update w/V for every coordinate.
    A changed prediction callback is explicitly used in the actual residual.
    """
    if not np.isfinite(y) or not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError('Finite target and positive learning rate are required.')
    x, w, V = np.asarray(x, float), np.asarray(w, float), np.asarray(V, float)
    prediction = predict_fn(x, w, V, bias)['prediction']
    error = float(y - prediction)
    weighted = V * x[:, None]
    gradient_w = x
    gradient_V = x[:, None] * (weighted.sum(axis=0) - weighted)
    new_w = w + learning_rate * error * gradient_w
    new_V = V + learning_rate * error * gradient_V
    if not np.isfinite(new_w).all() or not np.isfinite(new_V).all():
        raise ValueError('The update diverged; inspect features and the learning rate.')
    return {'w': new_w, 'V': new_V, 'bias': float(bias), 'prediction': prediction,
            'error': error, 'loss': .5*error**2}


### 본문 실행 정의 · `fm_epoch`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: 4298f6bb62a7dd4c923bba8781b57c5694ad67072d9f2c4aad57443b5fd9ab94
def fm_epoch(X, y, state: dict, order, learning_rate: float = .03,
             step_fn: Callable = embedding_fm_step, predict_fn: Callable = fm_score) -> dict:
    """Repeat the visible one-observation update; return new state, never mutate input."""
    current = {'w': state['w'].copy(), 'V': state['V'].copy(), 'bias': state['bias'],
               'updates': state.get('updates', 0)}
    for position in order:
        update = step_fn(X[position], y[position], current['w'], current['V'],
                         current['bias'], learning_rate, predict_fn=predict_fn)
        current = {'w': update['w'], 'V': update['V'], 'bias': update['bias'],
                   'updates': current['updates'] + 1}
    return current


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/09-feature-blocks.png" alt="사용자/제품 원핫과 두 리뷰 임베딩을 하나의 FM 특성으로 연결" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/09-feature-blocks.png) · 교재 개념을 재구성한 독자 도해입니다.


### P04 · 벡터를 숫자로 읽기
교재의 의미 임베딩 개념을 위한 **주어진 합성 벡터**입니다. u=(3,4), a=(4,3), b=(-4,3)의 길이·단위벡터·cos(u,a)·cos(u,b)를 먼저 구하세요. 이것은 실제 Nomic 출력이 아닙니다. 이후 실제 모델은768차원이며 앞d개를 골라 다시 정규화합니다. 이 선택은 PCA가 아닙니다.


In [ ]:
toy_vectors=np.array([[3.,4.],[4.,3.],[-4.,3.]])
normalized=normalize_embeddings(toy_vectors,2)
print('길이',np.linalg.norm(toy_vectors,axis=1),'단위벡터',normalized)
print('cos a/b',normalized[1:]@normalized[0])
my_P04=None


### 실제 훈련 리뷰만 인코딩·저장
train의 사용자/제품별 최근 비어 있지 않은 리뷰3개를 각300자까지 묶습니다. 선택·생략한 문자와 원본행ID를 추적합니다. 검증/시험 리뷰는 넣지 않습니다. 실제768차원 벡터에서 d=8을 선택하고 L2정규화합니다. 캐시는 텍스트·행ID·모델·d·집계/정규화 함수가 같을 때만 재사용합니다.


In [ ]:
recipe=make_feature_recipe(parts['train'],client,CACHE/'embeddings',dimension=8,
    review_fn=aggregate_reviews,normalize_fn=normalize_embeddings)
print('실제 인코딩/캐시:',recipe['metadata'])
training_rows=set(parts['train'].row_id)
assert all(set(ids)<=training_rows for group in recipe['source_rows'].values() for ids in group.values())
print('U,I,d:',len(recipe['users']),len(recipe['items']),recipe['dimension'])
sample=parts['train'].iloc[0]
x_real=assemble_features(sample.user_id,sample.item_id,recipe)
print('한 관측 x 크기',x_real.shape,'0이 아닌 칸',np.count_nonzero(x_real))


### P05 · 합성 특성 조립과 FM 수계산
U=2,I=3,d=1이면 n=7입니다. FM K=2이면 V는7×2입니다. 아래 x,w,V는 **손계산용으로 주어진 합성 수치**입니다. 선형항과 6개 활성 쌍의 상호작용을 계산하세요. 임베딩을 붙이면 원핫 두 칸의 쌍만 남는다는 가정은 성립하지 않습니다.


In [ ]:
toy=toy_embedding_fm()
toy_score=fm_score(toy['x'],toy['w'],toy['V'],toy['bias'])
print('x',toy['x'],'활성 V',toy['V'][toy['active']])
display(pd.DataFrame(pair_contributions(toy['x'],toy['V'])))
print('FM 분해',{k:v for k,v in toy_score.items() if k!='pairs'})
my_P05=None


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/10-fm-update.png" alt="같은 갱신 전 모수로 모든 기울기를 구한 뒤 SGD" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/10-fm-update.png) · 교재 개념을 재구성한 독자 도해입니다.


### P06 · 한 관측을 진짜 학습시키기
y=4, 예측4.6, L=0.5(y−예측)², α=.01일 때 error와 손실을 먼저 계산하세요. w의 첫 좌표는 얼마나 바뀌나요? V의 모든 좌표도 같은 갱신 전 값을 기준으로 바꿉니다. 편향은 이 실습에서 고정이며 L2는0입니다.


In [ ]:
update=embedding_fm_step(toy['x'],toy['y'],toy['w'],toy['V'],toy['bias'],.01,predict_fn=fm_score)
after=fm_score(toy['x'],update['w'],update['V'],toy['bias'])
print('오차/손실',update['error'],update['loss'],'새 w0',update['w'][0],'새 예측',after['prediction'])
my_P06=None


### 같은 분할·같은 학습 규칙으로 실제 FM 비교
기본 K=4,25epoch,α=.03. 훈련 평균만 편향으로 사용합니다. 같은 이름의 ID 특성은 동일하게 초기화하고 같은 관측 순서로 갱신합니다. 검증 RMSE가 최소인 w/V를 복원합니다. **시험 레이블은 fit에 쓰지 않습니다.** 훈련에 없는 제품은 원핫0·명시적 훈련평균 임베딩을 사용하고 건수를 표시합니다.


In [ ]:
models={}
for flag in [False,True]:
    name='임베딩+FM' if flag else 'ID만 FM'
    models[name]=fit_embedding_fm(parts,recipe,use_embeddings=flag,epochs=25,factors=4,learning_rate=.03,
        feature_fn=assemble_features,epoch_fn=fm_epoch,step_fn=embedding_fm_step,predict_fn=fm_score)
validation=pd.DataFrame([evaluate_embedding_fm(model,parts['validation']) for model in models.values()])
display(validation)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,3))
for name,model in models.items():
    ax.plot(model['history'].epoch,model['history'].validation_rmse,label=name)
ax.set(xlabel='epoch',ylabel='validation RMSE');ax.legend();plt.show()


### P07 · 한 조건을 수정하고 근거 남기기
d=8→4 또는 α=.03→.01 중 하나만 바꿉니다. 본문 함수의 동작도 한 번 수정해 보세요. 예를 들어 `assemble_features`에서 사용자 임베딩 블록만0으로 바꾸고 다시 학습하면 어떤 정보 경로가 사라질까요? 검증으로 선택하고 시험셋을 보며 다시 튜닝하지 않습니다. 캐시·갱신 횟수·미관측 수·RMSE를 기록하세요.


In [ ]:
my_P07=None


### 고정된 기본형의 보류 시험 · 한 번 읽기
아래 셀은 두 기본형의 최종 확인입니다. 위에서 선택을 바꾸었다면 설정과 선택 근거를 먼저 기록하세요. 전체 시험 점수와 미관측 수를 함께 읽습니다. 이 표는 무작위 관측 분할의 작은 표본이며 실제 미래 품질의 증거가 아닙니다.


In [ ]:
test_summary=pd.DataFrame([evaluate_embedding_fm(model,parts['test'],split_name='test') for model in models.values()])
display(test_summary)


### P08 · 앱 표·CSV·코드 연결
아래 앱은 학습된 w/V와 본문 특성·점수 함수를 함께 사용합니다. 사용자를 바꿔 점수의 세 항을 읽고 CSV를 받습니다. 새로운 코드를 정의한 뒤에는 모델을 다시 학습하고 앱을 다시 조립하세요. 예측은 범위를 자르지 않았으므로 1–5 밖일 수도 있습니다.


In [ ]:
from luna_recsys.llm_explainer import fm_view
selected_user=sorted(parts['validation'].user_id.unique())[0]
_,shown,state=fm_view(models['임베딩+FM'],parts['validation'],selected_user)
csv_path=export_displayed_table(state)
pd.testing.assert_frame_equal(pd.read_csv(csv_path),shown,check_dtype=False)
display(shown)
demo=build_llm_explainer(models=models,pairs=parts['validation'])
if IN_COLAB:
    demo.queue(default_concurrency_limit=1).launch(share=True,inline=False,prevent_thread_lock=True)
my_P08=None


In [ ]:
assert all(model['state']['updates']>0 for model in models.values())
assert np.isfinite(validation[['rmse','mae']].to_numpy()).all()
assert np.isfinite(test_summary[['rmse','mae']].to_numpy()).all()
assert set(parts['train'].row_id).isdisjoint(parts['test'].row_id)
print('W11B_EMBEDDING_FM_RUNTIME_PASS','실제 모델',EMBED_MODEL,'원차원',recipe['raw_dimension'],
      '특성차원',len(models['임베딩+FM']['feature_names']))
